# Prompting Qwen3.5-0.8B for a subset of CLARITY dataset

Reminder: We run on GPU.

In [1]:
!pip install -q -U "transformers @ git+https://github.com/huggingface/transformers.git"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 10.9 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 76.6 MB/s eta 0:00:00:00:01


We use a subset of our dataset and the small Qwen3.5-0.8B model.

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import torch
import time

model_name = "Qwen/Qwen3.5-0.8B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16, device_map="auto")

clarity_dataset = load_dataset("ailsntua/QEvasion", split="train[:300]")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

...

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.90M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/259k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3448 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/308 [00:00<?, ? examples/s]

## Creating system and user prompts

Note: It is up to you to write a better prompt.

In [3]:
system_prompt = """You are a political response classifier.
Given a question and its corresponding answer, classify the answer into one of the following categories:
- Clear Reply
- Ambivalent
- Clear Non-Reply

Respond with the label only."""

def build_prompt(example):
    return f"""Question: {example["question"]}
Answer: {example["interview_answer"]}"""

### Text Generation
model.generate()  runs autoregressive text generation — it takes an input token sequence and produces new tokens one at a time until a stopping condition is met.
### Parameters, based on the task and method
We define the max_new_tokens based on the output length we expect, and enforce greedy decoding using do_sample=False.

In [4]:
example = clarity_dataset[0]
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": build_prompt(example)}
]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

In [5]:
messages

[{'role': 'system',
  'content': 'You are a political response classifier.\nGiven a question and its corresponding answer, classify the answer into one of the following categories:\n- Clear Reply\n- Ambivalent\n- Clear Non-Reply\n\nRespond with the label only.'},
 {'role': 'user',
  'content': "Question: How would you respond to the accusation that the United States is containing China while pushing for diplomatic talks?\nAnswer: Well, look, first of all, theI am sincere about getting the relationship right. And one of the things that is going on now is, China is beginning to change some of the rules of the game, in terms of trade and other issues.And so one of the things we talked about, for example, is that they're now talking about making sure that no Chineseno one in the Chinese Government can use a Western cell phone. Those kinds of things.And so, really, what this trip was aboutit was less about containing China. I don't want to contain China. I just want to make sure that we hav

In [6]:
response

'Clear Non-Reply\n'

## Batched Inference

We run samples in **batches** to take advantage of GPU parallelism.

In [10]:
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

prompts = []
for example in clarity_dataset:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": build_prompt(example)}
    ]
    prompts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

BATCH_SIZE = 8
start = time.time()
results_batched = []

for i in range(0, len(prompts), BATCH_SIZE):
    batch_prompts = prompts[i:i + BATCH_SIZE]
    batch_examples = clarity_dataset.select(range(i, min(i + BATCH_SIZE, len(clarity_dataset))))
    inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    for j, example in enumerate(batch_examples):
        response = tokenizer.decode(outputs[j][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        results_batched.append({"question": example["question"], "response": response, "gold": example["clarity_label"]})

## Output Validation

We validate that model outputs fall within the expected label set: `{"Clear Reply", "Ambivalent", "Clear Non-Reply"}`.

In [11]:
VALID_LABELS = {"Clear Reply", "Ambivalent", "Clear Non-Reply"}

def sanity_check(results):
    total = len(results)
    valid = sum(1 for r in results if r["response"].strip() in VALID_LABELS)
    invalid = [r for r in results if r["response"].strip() not in VALID_LABELS]
    
    print(f"Valid outputs:   {valid}/{total}")
    print(f"Invalid outputs: {total - valid}/{total}")
    if invalid:
        print("\nInvalid responses:")
        for r in invalid:
            # print(f"  Q: {r['question'][:60]}...")
            print(f"  Response: '{r['response']}'")
            print()

In [12]:
sanity_check(results_batched)

Valid outputs:   299/300
Invalid outputs: 1/300

Invalid responses:
  Response: 'Yes
'



Continue computing the metrics, as you did in HW1 and HW2...